In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalogo_au")

In [0]:
catalogo = dbutils.widgets.get("catalogo")

In [0]:
spark.sql(f"DROP CATALOG IF EXISTS {catalogo} CASCADE;")


In [0]:
spark.sql(f"""CREATE CATALOG IF NOT EXISTS {catalogo}""")

In [0]:
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.raw;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.bronze;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.silver;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.golden;""")
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalogo}.exploratory;""")


###Tablas Bronze

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.bronze.spotify_tracks (
    track_id STRING,          -- Usaremos esto para 'rank'
    track_name STRING,
    artist_name STRING,
    album_name STRING,        -- Agregada: Buena práctica para datos de música
    popularity DOUBLE,        -- Para total_streams o streams_2025
    duration_ms LONG,         -- Agregada: Para convertir duration_seconds
    explicit BOOLEAN,
    release_date STRING,      -- Cambiado de year a date (más flexible)
    danceability DOUBLE,
    energy DOUBLE,
    ingestion_date TIMESTAMP
)
USING DELTA;
""")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.bronze.spotify_daily_charts (
  track_id STRING,
  rank INT,
  country STRING,
  snapshot_date DATE,
  ingestion_date TIMESTAMP)
USING delta
PARTITIONED BY (snapshot_date)
""")


###Tablas Silver

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.silver.tracks_transformed (
  track_id STRING,
  track_name STRING,
  artist_name STRING,
  popularity INT,
  popularity_category STRING,
  duration_min DOUBLE,
  release_year INT,
  energy_level STRING,
  danceability_level STRING,
  is_explicit_flag INT,
  ingestion_date TIMESTAMP)
USING DELTA;
""")



###Tablas Golden

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_au.golden.spotify_final_report (
  report_year INT,
  total_songs LONG,
  avg_popularity DOUBLE,
  top_artist STRING,
  energy_type STRING,
  is_mainstream STRING)
USING DELTA;
""")

